# TLFS23 — Setup

The TLFS23 dataset is hosted on Mendeley as a **single ~9 GB zipped folder** (`TLFS23 - Tamil Language Finger Spelling Image Dataset.zip`), already uploaded to my Google Drive. The full dataset has 248 classes with 1000 images each , which is too large to load directly from Drive in every notebook — Drive I/O quickly becomes a bottleneck in Colab.

Thus we run this notebook **once** to build a stratified subset on Drive (200 images/class, kept in a clean folder structure), then point every downstream notebook (EDA, preprocessing, EfficientNet-B3 training, Swin Transformer training) at that subset. The zip and this notebook are not needed again after Step 4 verifies cleanly.

**Pipeline:**
1. Mount Drive (where the zip lives)
2. Unzip the dataset to local Colab disk (`/content/...`) — fast scratch space, wiped at session end
3. Locate the `Dataset Folders` directory inside the unzipped tree
4. Sample 200 images/class and copy them to `/content/drive/MyDrive/TLFS23_subset/` on Drive
5. Verify all 248 classes are present and image count is correct

**Important:** do **not** delete the original zip until Step 5 confirms the subset is complete. The subset is what every other notebook in this project will load from.

## Step 0 — Mount Drive and point at the zip

Mount Google Drive so the zipped dataset is reachable at `/content/drive/MyDrive/...`. `ZIP_PATH` is the only thing that may need editing if the zip was renamed or placed in a subfolder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── ONLY CHANGE THIS if your zip has a different name ────────────────────────
ZIP_PATH = '/content/drive/MyDrive/TLFS23 - Tamil Language Finger Spelling Image Dataset.zip'

Mounted at /content/drive


## Step 1 — Unzip to local Colab disk

Unzipping straight into Drive would be painfully slow (Drive throttles small-file writes hard). Instead, extract to `/content/TLFS23/` on the Colab VM's local disk — this is fast scratch storage that disappears when the runtime ends, which is fine since we only need it long enough to build the subset.

Takes roughly 5 minutes.

In [ ]:
# ── Step 1: Unzip to local disk ───────────────────────────────────────────────
import os
from pathlib import Path

LOCAL_DIR = Path('/content/TLFS23')
LOCAL_DIR.mkdir(exist_ok=True)

print('Unzipping... (takes ~5 mins)')
os.system(f'unzip -q "{ZIP_PATH}" -d /content/TLFS23')
print('Unzip complete.')

# Show structure
!ls /content/TLFS23

Unzipping... (takes ~5 mins)
Unzip complete.
'TLFS23 - Tamil Language Finger Spelling Image Dataset'


## Step 2 — Locate the class folders

The zip extracts to a nested structure: `TLFS23 - Tamil Language Finger Spelling Image Dataset/Dataset Folders/`. That inner `Dataset Folders` directory is what holds the 248 per-class subdirectories (`1/`, `2/`, ..., `247/`, `Background/`). Confirm the count is 248 before proceeding — if it isn't, the path needs adjusting based on the `ls` output from Step 1.

In [ ]:
# ── Step 2: Find the Dataset Folders path ────────────────────────────────────
# Adjust if your folder structure differs
SRC_DIR = Path('/content/TLFS23/TLFS23 - Tamil Language Finger Spelling Image Dataset/Dataset Folders')

assert SRC_DIR.exists(), f'Source not found: {SRC_DIR}. Check output of ls above and update SRC_DIR.'

class_dirs = [d for d in SRC_DIR.iterdir() if d.is_dir()]
print(f'Source classes found: {len(class_dirs)}  (expected 248)')

Source classes found: 248  (expected 248)


## Step 3 — Build the stratified subset on Drive

For each of the 248 classes, randomly sample 200 images (seeded with `random.seed(42)` for reproducibility) and copy them into `/content/drive/MyDrive/TLFS23_subset/<class_name>/`. This gives a balanced ~49,600-image dataset that's small enough to live on Drive comfortably and load reasonably fast in downstream notebooks.

**Resume-safe:** if a class folder already has ≥200 images on Drive, it's skipped. So if Colab disconnects mid-copy (likely on a 248-class job over Drive), just re-run this cell and it picks up where it left off without redoing finished classes.

This is the slowest step — Drive writes are the bottleneck, expect this to take a while.

In [ ]:
# ── Step 3: Create subset on Drive (200 images/class) ────────────────────────
import shutil
import random

random.seed(42)

SUBSET_DIR     = Path('/content/drive/MyDrive/TLFS23_subset')
IMGS_PER_CLASS = 200
SUBSET_DIR.mkdir(parents=True, exist_ok=True)

img_exts = {'.jpg', '.jpeg', '.png', '.bmp'}
completed = 0

for cls_dir in sorted(SRC_DIR.iterdir(), key=lambda x: (x.name.isdigit() == False, int(x.name) if x.name.isdigit() else x.name)):
    if not cls_dir.is_dir():
        continue

    out_dir = SUBSET_DIR / cls_dir.name

    # Skip if already done (safe to re-run)
    if out_dir.exists() and len(list(out_dir.iterdir())) >= IMGS_PER_CLASS:
        completed += 1
        continue

    out_dir.mkdir(exist_ok=True)
    files  = [f for f in cls_dir.iterdir() if f.suffix.lower() in img_exts]
    sample = random.sample(files, min(IMGS_PER_CLASS, len(files)))
    for f in sample:
        shutil.copy(f, out_dir / f.name)

    completed += 1
    print(f'[{completed}/248] {cls_dir.name}', end='\r')

print(f'\nAll classes copied to Drive.')

[248/248] Background
All classes copied to Drive.


## Step 4 — Verify the subset is complete

Sanity check before trusting the subset: count class folders, count total images, and confirm no classes from `{1, 2, ..., 247, Background}` are missing. Only after this prints **"All 248 classes present"** is it safe to delete the original zip (if Drive space is tight).

From here on, every other notebook in the project should set:
```python
DATA_DIR = Path('/content/drive/MyDrive/TLFS23_subset')
```
and never touch the zip or this setup notebook again.

In [ ]:
# ── Step 4: Verify ────────────────────────────────────────────────────────────
class_dirs = [d for d in SUBSET_DIR.iterdir() if d.is_dir()]
total_imgs = sum(len(list(d.iterdir())) for d in class_dirs)

present  = {d.name for d in class_dirs}
expected = {str(i) for i in range(1, 248)} | {'Background'}
missing  = expected - present

print(f'Classes : {len(class_dirs)} / 248')
print(f'Images  : {total_imgs:,}  (expected 49,600)')

if missing:
    print(f'MISSING : {sorted(missing, key=lambda x: int(x) if x.isdigit() else 0)}')
else:
    print('All 248 classes present.')
    print()
    print('Setup complete! Point DATA_DIR in preprocessing notebook to:')
    print('  /content/drive/MyDrive/TLFS23_subset')

Classes : 248 / 248
Images  : 49,646  (expected 49,600)
All 248 classes present.

Setup complete! Point DATA_DIR in preprocessing notebook to:
  /content/drive/MyDrive/TLFS23_subset


## Step 5 — Quick sanity check (re-runnable later)

This cell is safe to run at the start of any future session to confirm the Drive subset is still intact and accessible — useful as the first cell to execute when opening a downstream notebook on a fresh Colab runtime.

In [ ]:
from pathlib import Path

DATA_DIR = Path('/content/drive/MyDrive/TLFS23_subset')
print(DATA_DIR.exists())
class_dirs = [d for d in DATA_DIR.iterdir() if d.is_dir()]
print(f'Classes found: {len(class_dirs)}')

True
Classes found: 248
